# Tratamento dos dados SONDA — Cachoeira Paulista

Este notebook lê **diretamente** os arquivos brutos `.dat` da pasta *Dados SONDA* (Google Drive),
visualiza todas as variáveis, aplica o controle de qualidade e gera a base **horária** usada nas
análises wavelet.

**Arquivos de origem** (formato TOA5, datalogger Campbell):

| Tipo | Conteúdo | Resolução |
|---|---|---|
| `CPA_AAAA_DDD_a_DDD_MD.dat` | meteorologia (temperatura, umidade, pressão, chuva, vento) | 10 min |
| `CPA_AAAA_DDD_a_DDD_SD.dat` | radiação (global, difusa, direta, onda longa, PAR, iluminância) | 1 min |

**Convenções verificadas:**
- horário em **UTC** (hora local = UTC−3). Conferido pelo meio-dia solar da radiação global, que cai a menos de 1 min do previsto para 45° W;
- o `TIMESTAMP` marca o **fim** do intervalo (o registro das 13:00 pertence à hora 12:00–13:00);
- a base horária é rotulada pelo **início** da hora.

As funções usadas estão em `src/dados/sonda.py`; os caminhos, em `config/caminhos_e_parametros.py`.

## 1. Configuração

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "config").is_dir())
sys.path.insert(0, str(RAIZ))

from config.caminhos_e_parametros import PASTA_FIGURAS_SONDA, PASTA_SONDA_TRATADA, pasta_dados_sonda
from src.dados import sonda

PASTA_SONDA = pasta_dados_sonda()
SALVAR_FIGURAS = True
EXPORTAR_XLSX = True
TROCA_DATALOGGER = pd.Timestamp("2023-12-01")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def salvar(fig, nome):
    if SALVAR_FIGURAS:
        PASTA_FIGURAS_SONDA.mkdir(parents=True, exist_ok=True)
        fig.savefig(PASTA_FIGURAS_SONDA / f"{nome}.png", dpi=200, bbox_inches="tight")


def rotulo(coluna):
    base = sonda.variavel_base(coluna)
    return f"{sonda.DESCRICOES.get(base, base)} [{coluna}]"


def marcar_eventos(eixo, coluna=None):
    eixo.axvline(TROCA_DATALOGGER, color="red", lw=1, ls="--")
    for inicio, fim in sonda.INTERVALOS_INVALIDOS.get(sonda.variavel_base(coluna or ""), []):
        eixo.axvspan(pd.Timestamp(inicio), pd.Timestamp(fim), color="red", alpha=0.12)


print("Dados brutos:", PASTA_SONDA)
print("Saida tratada:", PASTA_SONDA_TRATADA)

Dados brutos: G:\.shortcut-targets-by-id\16ch5d4P2kFqiX3nc1-Dd0zlMsHKTNDfV\Dados SONDA
Saida tratada: C:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\sonda_tratada


## 2. Inventário dos arquivos

Cada arquivo cobre cerca de um mês. A tabela mostra o período de cada arquivo, o datalogger e o
programa que o gerou. A linha vermelha tracejada nas figuras marca a **troca do datalogger
CR3000 → CR1000X em 01/12/2023**.

In [ ]:
inventario = sonda.listar_arquivos(PASTA_SONDA)
display(inventario.drop(columns="caminho"))
display(inventario.groupby(["tipo", "datalogger", "programa"]).size().rename("arquivos").to_frame())

sem_arquivo = sonda.dias_sem_arquivo(inventario.query("tipo == 'MD'"))
print("Dias sem arquivo:", ", ".join(sem_arquivo.strftime("%d/%m/%Y")) or "nenhum")

In [ ]:
fig, eixo = plt.subplots(figsize=(14, 2.6))
cores = {p: c for p, c in zip(sorted(inventario["programa"].unique()), ["#1b9e77", "#d95f02", "#7570b3"])}
for posicao, tipo in enumerate(sonda.TIPOS):
    for linha in inventario.query("tipo == @tipo").itertuples():
        eixo.barh(posicao, linha.data_fim - linha.data_inicio, left=linha.data_inicio,
                  color=cores[linha.programa], edgecolor="white")
eixo.set_yticks(range(len(sonda.TIPOS)), sonda.TIPOS)
eixo.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, label=p) for p, c in cores.items()],
            loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=8)
eixo.set_title("Cobertura dos arquivos por programa do datalogger")
salvar(fig, "inventario_arquivos")
plt.show()

## 3. Dicionário de variáveis

Unidades e tipo de processamento lidos do próprio cabeçalho dos arquivos. Na agregação horária:
`Avg`, `Std` e `WVc` → média (direção do vento → média circular), `Tot` → soma, `Max` → máximo,
`Min` → mínimo.

In [ ]:
dicionario = pd.concat(
    [sonda.dicionario_variaveis(PASTA_SONDA, t).assign(arquivo=t) for t in sonda.TIPOS],
    ignore_index=True,
)
dicionario["descricao"] = dicionario["variavel"].map(
    lambda v: sonda.DESCRICOES.get(sonda.variavel_base(v), "")
)
display(dicionario)

## 4. Leitura dos dados brutos

Todos os arquivos de cada tipo são concatenados. Registros com `TIMESTAMP` repetido (sobreposição
entre arquivos consecutivos) são removidos, mantendo o primeiro.

In [ ]:
brutos, relatorios_leitura = {}, {}
for tipo in sonda.TIPOS:
    brutos[tipo], relatorios_leitura[tipo] = sonda.carregar_sonda(PASTA_SONDA, tipo)
display(pd.DataFrame(relatorios_leitura).T)

## 5. Visualização dos dados brutos

### 5.1 Disponibilidade mensal

Percentual de registros válidos por mês e variável. Faixas escuras indicam variável ausente.

In [ ]:
def mapa_disponibilidade(dados, titulo, nome):
    mensal = dados.notna().resample("MS").mean() * 100
    fig, eixo = plt.subplots(figsize=(14, 0.28 * len(mensal.columns) + 1.6))
    imagem = eixo.imshow(mensal.T, aspect="auto", cmap="viridis", vmin=0, vmax=100, interpolation="nearest")
    eixo.set_yticks(range(len(mensal.columns)), mensal.columns, fontsize=8)
    meses = mensal.index.strftime("%m/%Y")
    eixo.set_xticks(range(0, len(meses), 2), meses[::2], rotation=90, fontsize=8)
    if TROCA_DATALOGGER in mensal.index:
        eixo.axvline(mensal.index.get_loc(TROCA_DATALOGGER) - 0.5, color="red", lw=1.5)
    eixo.grid(False)
    eixo.set_title(titulo)
    fig.colorbar(imagem, ax=eixo, label="% de registros validos", pad=0.01)
    salvar(fig, nome)
    plt.show()


for tipo in sonda.TIPOS:
    mapa_disponibilidade(brutos[tipo], f"Disponibilidade dos dados brutos - {tipo}", f"disponibilidade_bruto_{tipo}")

### 5.2 Séries brutas

Linha: média diária. Faixa: mínimo e máximo diários dos registros originais (10 min ou 1 min);
para a chuva, total diário. Edite `VARIAVEIS_VISUALIZAR` para ver outras colunas (por exemplo
`_std`, `_min`, `_max`).

Os picos absurdos que aparecem aqui (pressão de 602 mb, difusa de ±20 000 W/m², PAR de
190 000) são tratados na seção 6.

In [ ]:
VARIAVEIS_VISUALIZAR = {
    "MD": ["tp_sfc", "humid", "press", "rain", "ws10_avg", "wd10_avg"],
    "SD": ["glo_avg", "dif_avg", "dir_avg", "lw_raw_avg", "lw_calc_avg", "par_avg", "lux_avg",
           "tp_dir", "tp_lw_case"],
}


def plotar_series_diarias(dados, colunas, titulo, nome, referencia=None):
    fig, eixos = plt.subplots(len(colunas), 1, figsize=(14, 2.1 * len(colunas)), sharex=True)
    for eixo, coluna in zip(np.atleast_1d(eixos), colunas):
        serie = dados[coluna]
        if coluna == "rain":
            serie.resample("D").sum(min_count=1).plot(ax=eixo, color="#2166ac", lw=0.8)
        else:
            diario = serie.resample("D").agg(["mean", "min", "max"])
            eixo.fill_between(diario.index, diario["min"], diario["max"], color="#2166ac", alpha=0.18, lw=0)
            eixo.plot(diario.index, diario["mean"], color="#2166ac", lw=0.8)
        if referencia is not None and coluna in referencia:
            referencia[coluna].resample("D").mean().plot(ax=eixo, color="#999999", lw=0.6, zorder=0)
        marcar_eventos(eixo, coluna)
        eixo.set_title(rotulo(coluna), fontsize=9, loc="left")
        eixo.set_xlabel("")
    fig.suptitle(titulo, y=1.0)
    fig.tight_layout()
    salvar(fig, nome)
    plt.show()


for tipo in sonda.TIPOS:
    plotar_series_diarias(brutos[tipo], VARIAVEIS_VISUALIZAR[tipo], f"Dados brutos - {tipo}", f"series_bruto_{tipo}")

## 6. Controle de qualidade

Problemas encontrados na inspeção dos arquivos e as regras aplicadas (definidas em
`src/dados/sonda.py`; edite lá se quiser outro critério):

| Problema | Regra |
|---|---|
| `tilt_*` sem nenhum dado | coluna descartada |
| Pressão cai para ~602 mb em várias tardes (nov/2025–mar/2026): falha do sensor | `press` fora de 900–1000 mb → NaN |
| `ws10_max = 1000`, `wd10_min/max = 999`: códigos de falha | limites físicos do vento |
| Global e difusa ausentes ou com ±20 000 W/m² logo após a troca do datalogger (01/12/2023–07/02/2024) | intervalos inválidos → NaN |
| Rastreador solar parado até 12/03/2024: direta ~0 e difusa **igual à global** (esfera de sombreamento fora de posição) | direta e difusa inválidas até 12/03/2024 |
| `tp_lw_case` a −50 °C (jun–set/2023): termistor do pirgeômetro com defeito | `tp_lw_case` fora de −10–60 °C → NaN; `lw_calc` depende dele → NaN junto |
| PAR com razão PAR/global ~100 em vários meses (o físico é ~2 µmol/J) | `par` > 3·global + 20 → NaN |
| Umidade entre 100 e 105 % (3 363 registros, mais em mar–abr/2025 e 2026) | levada a 100 % (saturação do sensor) |

**Não tratado, precisa de confirmação com o orientador:**
- `humid`: o máximo da umidade **sobe aos poucos** (percentil 99,9: 96 % em 2023 → 99 % no fim de 2024 → 104 % em 2025), o que indica deriva de calibração do sensor, não só saturação. Levar a 100 % esconde essa deriva; vale perguntar se houve troca ou recalibração do sensor;
- `dir`: em mar–mai/2023 os máximos da direta caem para ~750 W/m² (contra ~950 nos outros meses), possível desalinhamento do rastreador;
- `par`: mesmo após o filtro, a série é intermitente (out/2024–abr/2025 e de jul/2025 em diante); pouco confiável;
- `lux`: ~1000 kLux ao meio-dia desde 12/2023 (sol pleno ≈ 100 kLux) → unidade ou fator de escala errado;
- 2023 (CR3000) não tem **chuva, vento, PAR nem lux**: essas variáveis só existem a partir de 12/2023.

In [ ]:
limpos, relatorios_qc = {}, {}
for tipo in sonda.TIPOS:
    limpos[tipo], relatorios_qc[tipo] = sonda.controle_qualidade(brutos[tipo])
limpos["MD"], umidade_saturada = sonda.saturar_umidade(limpos["MD"])

print(f"Umidade entre 100 e 105 % levada a 100 %: {umidade_saturada} registros")
for tipo in sonda.TIPOS:
    print(f"\nValores anulados por regra - {tipo}")
    display(relatorios_qc[tipo])
print("\nVariaveis suspeitas (mantidas, revisar):")
for variavel, motivo in sonda.VARIAVEIS_SUSPEITAS.items():
    print(f"  {variavel}: {motivo}")

### 6.1 Antes e depois

Cinza: dado bruto (máximo horário, para mostrar os picos). Azul: dado após o controle de qualidade.

In [ ]:
COMPARAR_QC = {"MD": ["press", "humid"], "SD": ["glo_avg", "dif_avg", "dir_avg", "par_avg", "lw_calc_avg"]}

pares = [(tipo, coluna) for tipo, colunas in COMPARAR_QC.items() for coluna in colunas]
fig, eixos = plt.subplots(len(pares), 1, figsize=(14, 2.2 * len(pares)), sharex=True)
for eixo, (tipo, coluna) in zip(eixos, pares):
    brutos[tipo][coluna].resample("h").max().plot(ax=eixo, color="#bbbbbb", lw=0.5, label="bruto")
    limpos[tipo][coluna].resample("h").max().plot(ax=eixo, color="#2166ac", lw=0.5, label="tratado")
    limite = sonda.LIMITES_FISICOS.get(sonda.variavel_base(coluna))
    if limite:
        eixo.set_ylim(limite[0] - 0.1 * (limite[1] - limite[0]), limite[1] + 0.1 * (limite[1] - limite[0]))
    marcar_eventos(eixo, coluna)
    eixo.set_title(rotulo(coluna), fontsize=9, loc="left")
    eixo.set_xlabel("")
eixos[0].legend(loc="upper right")
fig.suptitle("Controle de qualidade: bruto x tratado (eixo y limitado a faixa fisica)", y=1.0)
fig.tight_layout()
salvar(fig, "controle_qualidade_antes_depois")
plt.show()

## 7. Agregação horária

Horas com menos de 75 % dos registros esperados (6 no MD, 60 no SD) ficam vazias. A chuva horária
é a soma dos registros de 10 min.

In [ ]:
COBERTURA_MINIMA = 0.75

processamento = {
    tipo: sonda.ler_cabecalho(inventario.query("tipo == @tipo")["caminho"].iloc[0])["processamento"]
    for tipo in sonda.TIPOS
}
horarios = {}
for tipo in sonda.TIPOS:
    horarios[tipo], _ = sonda.agregar_horario(limpos[tipo], tipo, processamento[tipo], COBERTURA_MINIMA)

horario = horarios["MD"].join(horarios["SD"], how="outer")
print(f"Base horaria: {horario.shape[0]} horas x {horario.shape[1]} variaveis | "
      f"{horario.index.min():%d/%m/%Y %H:%M} a {horario.index.max():%d/%m/%Y %H:%M} UTC")
display(sonda.resumo_variaveis(horario).round(2))

In [ ]:
mapa_disponibilidade(horario, "Disponibilidade da base horaria tratada", "disponibilidade_horario")

## 8. Visualização dos dados tratados

### 8.1 Séries diárias

In [ ]:
principais = VARIAVEIS_VISUALIZAR["MD"] + VARIAVEIS_VISUALIZAR["SD"]
plotar_series_diarias(horario, principais, "Base horaria tratada (agregada por dia)", "series_tratado")

### 8.2 Ciclo diurno médio por mês

Média de cada hora do dia (hora **local**, UTC−3) em cada mês. Mostra como a amplitude do ciclo
diário muda ao longo do ano, que é a banda que a wavelet vai investigar.

In [ ]:
CICLO_DIURNO = ["tp_sfc", "humid", "press", "glo_avg", "lw_calc_avg", "ws10_avg"]

hora_local = (horario.index - pd.Timedelta(hours=3)).hour
mes = horario.index.to_period("M")
fig, eixos = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
for eixo, coluna in zip(eixos.flat, CICLO_DIURNO):
    tabela = horario[coluna].groupby([hora_local, mes]).mean().unstack()
    imagem = eixo.pcolormesh(np.arange(tabela.shape[1] + 1), np.arange(25), tabela.to_numpy(), cmap="turbo", shading="flat")
    eixo.set_title(rotulo(coluna), fontsize=9)
    passo = 6
    eixo.set_xticks(np.arange(0, tabela.shape[1], passo) + 0.5, tabela.columns.strftime("%m/%y")[::passo], rotation=90, fontsize=8)
    fig.colorbar(imagem, ax=eixo, pad=0.01)
for eixo in eixos[:, 0]:
    eixo.set_ylabel("Hora local")
fig.tight_layout()
salvar(fig, "ciclo_diurno_mensal")
plt.show()

## 9. Lacunas

Maiores lacunas da base horária. A wavelet exige série contínua, então lacunas longas definem
quais trechos podem ser analisados sem interpolação pesada.

In [ ]:
LACUNAS_VERIFICAR = ["tp_sfc", "humid", "press", "rain", "glo_avg", "dif_avg", "dir_avg", "lw_calc_avg", "par_avg"]

tabelas = []
for coluna in LACUNAS_VERIFICAR:
    tabela = sonda.lacunas(horario[coluna], pd.Timedelta(hours=1))
    tabelas.append(pd.Series({
        "lacunas": len(tabela),
        "horas_faltantes": tabela["duracao_h"].sum(),
        "maior_lacuna_h": tabela["duracao_h"].max() if len(tabela) else 0,
        "inicio_maior": tabela["inicio"].iloc[0] if len(tabela) else pd.NaT,
        "lacunas_ate_3h": int((tabela["duracao_h"] <= 3).sum()),
    }, name=coluna))
resumo_lacunas = pd.DataFrame(tabelas)
display(resumo_lacunas)

## 10. Exportação

Arquivos gerados em `PASTA_SONDA_TRATADA` (pasta `2026/sonda_tratada`):

- `sonda_horario.csv`: base horária tratada, todas as variáveis, horário UTC rotulado pelo início da hora;
- `dicionario_variaveis.csv`, `relatorio_controle_qualidade.csv`, `lacunas.csv`;
- `sonda_horario.xlsx` (se `EXPORTAR_XLSX = True`): as mesmas tabelas em abas, para consulta no Excel.

In [ ]:
PASTA_SONDA_TRATADA.mkdir(parents=True, exist_ok=True)
relatorio_qc = pd.concat(relatorios_qc, names=["arquivo"]).reset_index()

horario.to_csv(PASTA_SONDA_TRATADA / "sonda_horario.csv", float_format="%.4f")
dicionario.to_csv(PASTA_SONDA_TRATADA / "dicionario_variaveis.csv", index=False)
relatorio_qc.to_csv(PASTA_SONDA_TRATADA / "relatorio_controle_qualidade.csv", index=False)
resumo_lacunas.to_csv(PASTA_SONDA_TRATADA / "lacunas.csv")

if EXPORTAR_XLSX:
    with pd.ExcelWriter(PASTA_SONDA_TRATADA / "sonda_horario.xlsx") as escritor:
        horario.to_excel(escritor, sheet_name="horario_utc")
        dicionario.to_excel(escritor, sheet_name="dicionario", index=False)
        relatorio_qc.to_excel(escritor, sheet_name="controle_qualidade", index=False)
        resumo_lacunas.to_excel(escritor, sheet_name="lacunas")

for arquivo in sorted(PASTA_SONDA_TRATADA.iterdir()):
    print(f"{arquivo.name:40s} {arquivo.stat().st_size / 1e6:6.1f} MB")